# ECO462: Homework 5

In [83]:
# Import basic Python libraries
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import norm 

## Section A: $M^2$ Measure of Portfolio Performance
*See all the graphs in the handwritten portion above*

### Question A1

The article defines the $M^2$ measure to be:

<center>"... the return an investor would have earned in a particular period if the fund had been diluted or leveraged to match the benchmark's risk."</center>

Therefore, first employ modern portfolio theory and envision an efficient frontier and a matching market portfolio. Assume that the market portfolio is the benchmark in this case, and that each other fund will be compared to the market portfolio's risk to calculate the $M^2$ figure. By connecting the risk free rate $r_{rf}$ to the market portfolio point $M$, you create the CML which will thusforth serve as the basis of comparison.

Now, select any fund and create a CAL by connecting the fund's point on the graph to the risk free rate. Most likely, the fund will not have the same risk (equivalently defined as variance) and thus needs to be normalized. In order to accomplish this, find the slope of the CAL (which happens to be the fund's Sharpe ratio) and multiply it by the market portfolio's variance $V[r_M]$ in order for the risk's to be the same. The corresponding return at the x-value $V[r_M]$ is the $M^2$ value.

As shown in the second graph with portfolios $A$ and $B$ on the CAL, portfolio $A$ which is fully weighted in the original fund and then is "diluted" to match the benchmark risk by adding weight in the MM. Hence, the weight $\omega _A$ is lessened and becomes less than $1$ while $1-\omega _A$ is increased above 0 until the variance of the final portfolio $B$ matches the market portfolio.

### Question A2

Again, assume the "benchmark" decribed in the definition above is the market portfolio. The $M^2$ measure for the market portfolio is simply the unit measure $1$ since the $M^2$ measure inherently has the same level of the benchmark since it *is* the benchmark.

### Question A3

The $M^2$ measures of all the different funds that plot on the CML are also the unit measure $1$ since all assets on the CML have the same Sharpe ratio and therefore the same expected-return-risk tradeoff as the market. Consider the method to acquire the $M^2$ ratio is finding the Sharpe ratio and multiplying that Sharpe ratio by the market portfolio's variance, thus since both variables of the formula would be identical, so would the $M^2$ measure.

### Question A4

The $M^2$ meausres will be lower than the funds that plot on the ML since their Sharpe ratios will be lower, hence when multiplied by the same variance, the expected return will be below the CML.


### Question A5

The $M^2$ measure is given by the expected return of a particular fund on the same CAL because the $M^2$ measure is simply the Sharpe ratio multipled by the benchmark / market portfolio's variance, as mentioned in previous part. Simplying by changing the weights, either "diluting" or "leveraging" the fund, each asset will eventually land on porfolio $B$ as shown by the graph. By definition, every asset found on the same CAL will have the same Sharpe ratio and therefore the same $M^2$ since both terms of the formula would be identical. 

Therefore given any asset on the same CAL, the $M^2$ measure will be known for all other assets found on the same CAL with no further information. If the specific weights of adjustment are desired, then more information would be needed.

### Question A6

Some of these funds contain assets other than stocks and bonds because they have a Sharpe ratio and therefore $M^2$ ratio higher than the market. According to modern portfolio theory, the optimal portfolio contains some weight $\omega$ of the market portfolio and some weight $1-\omega$ of the MM account which will offer the highest Sharpe ratio. The modern portfolio theory is contradicted by the higher return of some of the funds with higher Sharpe ratio and $M^2$ measure, hence there must be other assets that allow the corresponding asset mixture to dominate the set of "optimal" portfolios that contain only the market portfolio and the MM.


### Question A7

Reasonably, no, since the funds or assets on the same CAL have the same Sharpe Ratio and correspondingly the same $M^2$. 

However, this follows under the assumption of accessible capital and leverage. Tn pathological cases of restricted borrowing or inaccessibility of leverage, then "diluting" and "leveraging" may not be possible and it may be feasible to not have a straight CAL. 

For instance, consider the graph where there is no borrowing. While normally while this is possible by mixing a portfolio with other assets in the given collection of efficient frontier assets, the $M^2$ measure restricts to only borrowing the cash at the rist free rate or leveraging the fund. Hence, there is no way to get to the volatility of the market and the $M^2$ would be undefined $note the dashed orange line on the graph in which it would be required to traverse to achieve a volatility of the market which cannot be traversed without borrowing).

### Question A8

First, calculate the expected return of the weighted linear combination of the expected return of the stock market portfolio P and cash, such that the overall expected return $\mathbb{E}[r_F]$ is:

$$\mathbb{E}[r_F] = \omega \mathbb{E}[r_M]-(1-\omega)r_{rf}$$

Then, you solve for the Sharpe ratio (equivalently the slope of the CAL in which your new combined portfolio $F$ lies on) of $F$:

$$\dfrac{\mathbb{E}[r_F]-r_{rf}}{V[r_p]}$$

Then, to find the $M^2$ measure, multiply the Sharpe ratio of $F$ by the variance of the market portfolio:

$$M^2 = \dfrac{\mathbb{E}[r_F]-r_{rf}}{V[r_p]}\times V[r_M]\quad=\quad\dfrac{(\omega \mathbb{E}[r_M]-(1-\omega)r_{rf})-r_{rf}}{V[r_p]}\times V[r_M]$$

## Section B: Fund Performance Metrics and Analysis

In [107]:
import pandas as pd
import statsmodels.api as sm

df_funds = pd.read_excel("../data/FUNDS.xlsx")

df_funds.columns = df_funds.iloc[0]
df_funds = df_funds.iloc[1:]
df_funds["Date"] = pd.to_datetime(df_funds["Date"])
df_funds = df_funds.set_index("Date")

df_funds = df_funds.apply(pd.to_numeric, errors="coerce")

df_funds["TBILL_weekly"] = (1 + df_funds["TBILL"]) ** (1 / 52) - 1

df_funds["SP500D_return"] = df_funds["SP500D"].pct_change() * 100

excluded = {"TBILL", "TBILL_weekly", "SP500D", "SP500D_return"}
fund_cols = [c for c in df_funds.columns if c not in excluded]

df_returns = df_funds.dropna(subset=["TBILL_weekly", "SP500D_return"] + fund_cols).copy()

df_returns = df_returns.drop(columns=["TBILL", "SP500D"], errors="ignore")
df_returns = df_returns.rename(columns={"TBILL_weekly": "TBILL", "SP500D_return": "SP500D"})

sample_cutoff = pd.to_datetime("2011-01-01")
df_pre = df_returns.loc[:sample_cutoff - pd.Timedelta(days=1)]
df_post = df_returns.loc[sample_cutoff:]

def market_model(asset, data):
    df = pd.DataFrame(index=data.index)
    df["Excess Asset Return"] = data[asset] - data["TBILL"]
    df["Excess Market Return"] = data["SP500D"] - data["TBILL"]
    return df

def run_sample(data):
    results = []
    for t in fund_cols:
        df = market_model(t, data)
        X = sm.add_constant(df["Excess Market Return"])
        y = df["Excess Asset Return"]
        model = sm.OLS(y, X, missing="drop").fit()

        results.append({
            "Fund": t,
            "Alpha": model.params["const"],
            "Beta": model.params["Excess Market Return"],
            "R²": model.rsquared,
            "p(Alpha=0)": model.pvalues["const"]
        })
    return pd.DataFrame(results).set_index("Fund")

print("=========== Full Sample Market Model ===========")
print(run_sample(df_returns).round(4))

print("\n=========== Pre-2011 Market Model ===========")
print(run_sample(df_pre).round(4))

print("\n=========== Post-2011 Market Model ===========")
print(run_sample(df_post).round(4))


=========== Full Sample Market Model ===========
        Alpha    Beta      R²  p(Alpha=0)
Fund                                     
CPODX  0.0061  1.2850  0.5936      0.9370
WSHFX  0.0181  0.8751  0.9214      0.3263
TMSIX  0.0471  1.0730  0.8659      0.1233
ACEIX  0.0369  0.6641  0.8911      0.0281
JABAX  0.0477  0.5402  0.8711      0.0015

=========== Pre-2011 Market Model ===========
        Alpha    Beta      R²  p(Alpha=0)
Fund                                     
CPODX  0.0473  1.3466  0.6501      0.6673
WSHFX  0.0281  0.8480  0.8949      0.3852
TMSIX  0.1160  1.0219  0.8569      0.0128
ACEIX  0.0852  0.6169  0.9071      0.0001
JABAX  0.0699  0.4871  0.8203      0.0060

=========== Post-2011 Market Model ===========
        Alpha    Beta      R²  p(Alpha=0)
Fund                                     
CPODX -0.0150  1.1981  0.5165      0.8888
WSHFX -0.0007  0.9145  0.9593      0.9658
TMSIX -0.0388  1.1492  0.8840      0.3142
ACEIX -0.0270  0.7341  0.8898      0.2593
JABAX  0.0081  0

### Question B1

Over the full sample, the estimated market model indicates that the alphas (Jensen’s measures of abnormal performance) are mostly small and statistically insignificant. Using the regression model
$ R_{it} - R_{ft} = \alpha_i + \beta_i (R_{mt} - R_{ft}) + \varepsilon_{it} $,
the null hypothesis $ H_0: \alpha_i = 0 $ tests for no superior performance. Across all five funds, only ACEIX and JABAX show significant positive alphas, with $ \alpha_{ACEIX} = 0.0369 $ ($ p = 0.0281 $) and $ \alpha_{JABAX} = 0.0477 $ ($ p = 0.0015 $). Their corresponding t-statistics are roughly 2.2 and 3.2, respectively, confirming outperformance at the 5% and 1% levels. The remaining funds (CPODX, WSHFX, and TMSIX) have p-values greater than 0.1, suggesting that their observed positive alphas could be due to noise. Betas range from 0.54 for JABAX to 1.29 for CPODX, indicating that JABAX carries substantially less systematic risk, while CPODX is slightly aggressive relative to the market. Overall, for the full period, only ACEIX and JABAX exhibit statistically significant risk-adjusted outperformance, while the other funds behave in line with market expectations.

When the sample is split into two subperiods, clear temporal differences emerge. In the pre-2011 period (June 5, 1999–Dec 25, 2010), three of the five funds display strong evidence of superior performance. TMSIX shows $ \alpha = 0.1160 $ with $ p = 0.0128 $ (t ≈ 2.5), ACEIX exhibits $ \alpha = 0.0852 $ with $ p = 0.0001 $ (t ≈ 3.9), and JABAX has $ \alpha = 0.0699 $ with $ p = 0.0060 $ (t ≈ 2.7). Each of these alphas is positive and statistically significant, rejecting the null hypothesis of no abnormal return at conventional levels. CPODX and WSHFX, however, again fail to show significance ($ p = 0.67 $ and $ p = 0.39 $, respectively). In this early period, several funds clearly outperform on a risk-adjusted basis, suggesting either managerial skill or market conditions conducive to generating alpha. The $ R^2 $ values are relatively high (0.82–0.91 for most funds), indicating a strong explanatory power of the market model. Betas remain below unity for all except CPODX, which continues to show high market sensitivity ($ \beta \approx 1.35 $).

In contrast, in the post-2011 period (Jan 1, 2011–Aug 20, 2022), none of the estimated alphas are statistically different from zero. CPODX turns slightly negative with $ \alpha = -0.0150 $ ($ p = 0.8888 $), WSHFX is virtually zero ($ \alpha = -0.0007, p = 0.9658 $), TMSIX’s alpha becomes negative ($ \alpha = -0.0388, p = 0.3142 $), and ACEIX’s and JABAX’s alphas also lose their earlier significance ($ \alpha = -0.0270, p = 0.2593 $ and $ \alpha = 0.0081, p = 0.5422 $, respectively). The t-statistics for these alphas are all near zero, ranging from roughly –1.1 to 0.6. In short, after 2011, every fund’s abnormal return is statistically indistinguishable from zero, so the null hypothesis $ H_0: \alpha_i = 0 $ cannot be rejected for any of them. Importantly, betas remain stable relative to the earlier period—most funds’ $ \beta $ estimates change by less than 0.1—so the change in performance is not due to increased market exposure but to a decline in excess returns.

Comparing the two subperiods highlights a consistent pattern of alpha deterioration over time. For instance, TMSIX’s alpha falls from $ 0.1160 $ to $ -0.0388 $, a decline of roughly 0.15 per period; ACEIX drops from $ 0.0852 $ to $ -0.0270 $ (a 0.11 reduction); and JABAX declines from $ 0.0699 $ to $ 0.0081 $ (a 0.06 reduction). The t-statistics associated with these alphas move from 2–4 in the pre-2011 sample to approximately zero afterward, reinforcing the conclusion that the evidence of superior performance evaporates in the later period. This loss of statistical significance could stem from heightened market efficiency, structural shifts in fund management, or the growing difficulty of consistently outperforming broad indices in the post-crisis, low-rate environment.

In summary, the full-sample and pre-2011 analyses provide some support for superior risk-adjusted performance in a subset of funds—most notably ACEIX, JABAX, and to a lesser extent TMSIX—while the post-2011 results reveal no fund with a statistically significant alpha. The t-tests clearly indicate that the null of no superior performance cannot be rejected in the recent period for any fund. Thus, the overall assessment changes: while these funds once demonstrated measurable skill or advantage, this performance does not persist. By the post-2011 sample, their returns appear fully explained by market exposure ($ \beta $) alone, with $ \alpha_i \approx 0 $ and $ p > 0.25 $ in every case, consistent with efficient-market behavior and the erosion of persistent alpha over time.

### Question B2

<u>__***CPODX***__</u>: The **adjusted expense ratio** is .9%, which is equivalent to the **non-adjusted expense ratio**, and the **distribution fee level** is high (an evaluation of a mutual fund share class' expense ratio relative to other funds that invest in similar asset classes).

The **investment style** is mid-growth and the **category** is large growth, with a *44% turnover*.

The **strategy** is long-term capital appreciation by investing primarily in established and emerging companies, with an emphasis on a bottom-up stock selection process.

The **TTM yield** (trailing 12-month yield), which is a backward-looking metric showing the total income, dividends and interest, distributed by the fund to investors over the past 12 months, was .48%.

<u>__***WSHFX***__</u>: The **adjusted expense ratio** is .630%, which is slightly higher than the **non-adjusted expense ratio** of .620%, and the **distribution fee level** is below average.

The **investment style** is large blend and the **category** is large value, with a *29% turnover*.

The **strategy** is to produce income and to provide an opportunity for growth of principal consistent with sound common stock investing, maintaining a fully invested, diversified portfolio, consisting primarily of high-quality common stocks that have a strong record of earnings and dividends.

The **TTM yield** is 1.14%.

<u>__***TMSIX***__</u>: The **adjusted expense ratio** is .740%, which is equivalent to the **non-adjusted expense ratio**, and the **distribution fee level** is below average.

The **investment style** is mid blend and the **category** is mid-cap blend, with a *42% turnover*.

The **strategy** is long-term capital appreciation by investing at least 80% of its net assets, plus the amount of any borrowing for investment purposes, in equity securities of mid-sized companies.

The **TTM yield** is 0.49%.

<u>__***ACEIX***__</u>: The **adjusted expense ratio** is .770%, which is equivalent to the **non-adjusted expense ratio**, and the **distribution fee level** is low.

The **investment style** is large value and the **category** is moderate allocation, with a *139% turnover*.

The **strategy** seeks current income and, secondarily, capital appreciation, normally investing at least 80% of its net assets, plus any borrowings for investment purposes, in equity and income securities, and in derivatives and other instruments that have economic characteristics similar to such securities. 

It invests, under normal circumstances, at least 65% of its net assets in income-producing equity investments. 

The fund may invest up to 25% of its net assets in securities of foreign issuers.

The **TTM yield** is 1.82%.

<u>__***JABAX***__</u>: The **adjusted expense ratio** is .820%, which is slightly higher than the **non-adjusted expense ratio** of .800%, and the **distribution fee level** is average.

The **investment style** is large blend and the **category** is moderate allocation, with a *76% turnover*.

The **strategy** seeks long-term capital growth, consistent with preservation of capital and balanced by current income. 

The fund pursues its investment objective by normally investing 35‑70% of its assets in equity securities and the remaining assets in fixed-income securities and cash equivalents. 

It normally invests at least 25% of its assets in fixed-income senior securities. The fund may also invest in money market instruments.

The fund will limit its investments in high-yield/high-risk bonds to 35% of the fixed-income portion of its net assets.

The **TTM yield** is 1.58%.

Yes, the qualitative information about each mutual fund aligns well with the statistical results from the market model regressions. The descriptive data on expenses, turnover, yield, and strategy help explain the alphas’ signs and significance, showing that funds with higher costs or conservative mandates tended not to produce statistically significant abnormal returns, while more flexible or mid-cap funds briefly achieved alpha before 2011 but not afterward.

For **CPODX**, the lack of significant alpha across all periods ($p = 0.937$ full sample, $p = 0.667$ pre-2011, $p = 0.889$ post-2011) is consistent with its high 0.9% expense ratio and high distribution fee level. As a mid-growth fund in the large-growth category, it operates in one of the most efficient segments of the market, limiting opportunities for persistent outperformance. Its moderate turnover (44%) and very low yield (0.48%) indicate a focus on capital gains rather than income, but the high fees likely offset any stock-selection benefit, explaining its statistically insignificant alpha.

**WSHFX** also shows no significant alpha ($p = 0.326$ full sample, $p = 0.385$ pre-2011, $p = 0.966$ post-2011), which aligns with its conservative, income-oriented large-value approach. The low turnover (29%), below-average distribution fees, and reasonable expense ratio (0.63%) reflect a steady, market-tracking strategy. The higher yield (1.14%) and a beta near 0.9 indicate the fund behaves similarly to a defensive large-cap portfolio. Hence, its insignificant alpha is consistent with its design—steady income and diversification rather than active excess-return generation.

For **TMSIX**, the strong pre-2011 alpha ($\alpha = 0.1160$, $p = 0.0128$) but insignificance afterward ($p = 0.3142$) match its mid-cap blend profile. The below-average expenses (0.74%) and moderate turnover (42%) allowed for active exposure to mid-cap inefficiencies, which likely contributed to earlier outperformance. As mid-caps became more efficiently priced after 2011, that advantage disappeared, leaving no significant abnormal return. Its low yield (0.49%) confirms its focus on price appreciation, consistent with its earlier but temporary alpha.

**ACEIX**’s strong and significant alpha before 2011 ($p = 0.0001$) and insignificance afterward ($p = 0.2593$) fit with its aggressive, high-turnover (139%) equity-income strategy. The low distribution fees and moderate expenses (0.77%) likely enabled efficient active management, explaining its earlier outperformance. Its higher yield (1.82%) and flexible, moderate-allocation profile also match the fund’s dual focus on income and capital gains. The later loss of alpha aligns with reduced opportunities for active managers in income-oriented mixed portfolios post-2011.

**JABAX** similarly transitions from significant alpha pre-2011 ($p = 0.006$) to insignificance post-2011 ($p = 0.542$). Its balanced allocation—35-70% equities and the remainder in fixed income—along with moderate turnover (76%) and average expenses (0.82%) explains its earlier ability to generate alpha through tactical shifts. The 1.58% yield and diversified profile emphasize stability rather than aggressive alpha pursuit, matching its post-2011 convergence toward market-level performance.

Overall, the descriptive data confirm the quantitative results: funds with higher costs or passive styles (CPODX, WSHFX) show no persistent abnormal returns, while those with more flexible, active, or mid-cap strategies (TMSIX, ACEIX, JABAX) achieved significant alpha only during earlier, less efficient periods. This consistency reinforces that expense structures, market segment, and strategy type all align with the observed performance patterns in the regression results.

### Question B3 

The question was skipped and announced to be optional.

In [ ]:
# Question B4 Coding Portion

def performance_ratios(returns, regression_results):
    ratios = []
    for fund, row in regression_results.iterrows():
        beta = row["Beta"]
        alpha = row["Alpha"]

        # Compute mean and standard deviation of excess returns
        excess_returns = returns[fund] - returns["TBILL"]
        mean_excess = excess_returns.mean()
        std_excess = excess_returns.std(ddof=1)

        # Treynor ratio: (E[r_i - r_f]) / β_i
        treynor = mean_excess / beta if beta != 0 else np.nan

        # Sharpe ratio: (E[r_i - r_f]) / σ(r_i - r_f)
        sharpe = mean_excess / std_excess if std_excess != 0 else np.nan

        ratios.append({
            "Fund": fund,
            "Alpha (Jensen)": alpha,
            "Beta": beta,
            "Mean Excess Return": mean_excess,
            "Std Excess Return": std_excess,
            "Treynor Ratio": treynor,
            "Sharpe Ratio": sharpe
        })

    return pd.DataFrame(ratios).set_index("Fund")

# --- Compute ratios for each sample ---
print("\n=========== FULL SAMPLE PERFORMANCE RATIOS ===========")
results_full = run_sample(df_returns)
df_perf_full = performance_ratios(df_returns, results_full)
print(df_perf_full.round(4))

print("\n=========== PRE-2011 PERFORMANCE RATIOS ===========")
results_pre = run_sample(df_pre)
df_perf_pre = performance_ratios(df_pre, results_pre)
print(df_perf_pre.round(4))

print("\n=========== POST-2011 PERFORMANCE RATIOS ===========")
results_post = run_sample(df_post)
df_perf_post = performance_ratios(df_post, results_post)
print(df_perf_post.round(4))


=========== FULL SAMPLE PERFORMANCE RATIOS ===========
       Alpha (Jensen)    Beta  Mean Excess Return  Std Excess Return  \
Fund                                                                   
CPODX          0.0061  1.2850              0.1982             4.1847   
WSHFX          0.0181  0.8751              0.1490             2.2875   
TMSIX          0.0471  1.0730              0.2075             2.8931   
ACEIX          0.0369  0.6641              0.1362             1.7653   
JABAX          0.0477  0.5402              0.1285             1.4522   

       Treynor Ratio  Sharpe Ratio  
Fund                                
CPODX         0.1542        0.0474  
WSHFX         0.1702        0.0651  
TMSIX         0.1934        0.0717  
ACEIX         0.2051        0.0771  
JABAX         0.2379        0.0885  

=========== PRE-2011 PERFORMANCE RATIOS ===========
       Alpha (Jensen)    Beta  Mean Excess Return  Std Excess Return  \
Fund                                                   

### Question B4 Coding Portion

In addition to the Jensen index, two other important measures of risk-adjusted performance are the **Treynor ratio** and the **Sharpe ratio**. These indices help evaluate how well each fund compensates investors for the level of risk taken, but they differ in how they define that risk.

The **Treynor ratio** measures the excess return earned per unit of *systematic risk*, represented by the fund’s beta. It is calculated as:

$$
T_i = \frac{\bar{r_i} - \bar{r_f}}{\beta_i}
$$

where $\bar{r_i}$ is the average return on fund *i*, $\bar{r_f}$ is the average risk-free rate (in this case, the T-bill rate), and $\beta_i$ is the estimated beta from the market model regression. A higher Treynor ratio indicates that the fund generates more return per unit of market risk exposure, implying superior performance for a well-diversified portfolio.

The **Sharpe ratio**, on the other hand, measures excess return per unit of *total risk*, captured by the standard deviation of excess returns. It is given by:

$$
S_i = \frac{\bar{r_i} - \bar{r_f}}{\sigma_i}
$$

where $\sigma_i$ is the standard deviation of the fund’s excess returns. Unlike the Treynor ratio, the Sharpe ratio penalizes funds that are not well-diversified since it incorporates both systematic and unsystematic risk.

Finally, the **Jensen index** (or Jensen’s alpha) measures a fund’s abnormal return relative to what would be expected based on its beta and the market’s excess return. A positive alpha suggests that the fund manager has added value through active management beyond market movements.

When comparing the results across these three measures, the **rankings of the five funds may be similar but not identical**. If the funds are well-diversified, the Treynor and Sharpe ratios should produce similar rankings because unsystematic risk is minimal. However, if some funds carry more idiosyncratic risk, the Sharpe ratio—being sensitive to total volatility—may rank them lower than the Treynor ratio. Jensen’s alpha may also differ if certain funds consistently outperform or underperform relative to their expected risk-adjusted benchmark. Overall, consistency across all three measures would indicate stable and well-diversified performance among the funds.

### Question B5 

Consider the extended market model that is used for market timing:
$$r_{Pt} - r_{ft} = \alpha_P + \beta_P \{ r_{Mt} - r_{ft} \} + \gamma_P \{ r_{Mt} - r_{ft} \}^2 + \varepsilon_{Pt}$$

Then run regression again on the full sample, with the code shown below.

In [112]:
# Question B5 Coding Portion

def extended_market_model(asset, returns):
    df_asset = pd.DataFrame(index=returns.index)
    df_asset["Excess_Asset_Returns"] = returns[asset] - returns["TBILL"]
    df_asset["Excess_Market_Returns"] = returns["SP500D"] - returns["TBILL"]
    df_asset["Excess_Market_Returns_Sq"] = df_asset["Excess_Market_Returns"] ** 2
    return df_asset

def run_extended_regression(df_returns_subset):
    tickers = df_returns_subset.columns[:-2]
    results = []

    for t in tickers:
        df_stock = extended_market_model(t, df_returns_subset)
        X = sm.add_constant(df_stock[["Excess_Market_Returns", "Excess_Market_Returns_Sq"]])
        y = df_stock["Excess_Asset_Returns"]

        model = sm.OLS(y, X, missing="drop").fit()

        gamma = model.params["Excess_Market_Returns_Sq"]
        se_gamma = model.bse["Excess_Market_Returns_Sq"]
        t_gamma = gamma / se_gamma
        p_gamma_one_sided = 1 - norm.cdf(t_gamma)

        results.append({
            "Stock": t,
            "Alpha (Selectivity)": model.params["const"],
            "Beta": model.params["Excess_Market_Returns"],
            "Gamma (Timing)": gamma,
            "t(Gamma>0)": t_gamma,
            "p(Gamma>0)": p_gamma_one_sided,
            "R²": model.rsquared,
            "p(Alpha=0)": model.pvalues["const"]
        })
    return pd.DataFrame(results).set_index("Stock")

print("=========== FULL SAMPLE: Extended Market Model (Market Timing) ===========")
df_full = run_extended_regression(df_returns)
print(df_full.round(4))

=========== FULL SAMPLE: Extended Market Model (Market Timing) ===========
       Alpha (Selectivity)    Beta  Gamma (Timing)  t(Gamma>0)  p(Gamma>0)  \
Stock                                                                        
CPODX              -0.0849  1.3010          0.0140      3.1874      0.0007   
WSHFX               0.0293  0.8732         -0.0017     -1.6192      0.9473   
TMSIX               0.0400  1.0742          0.0011      0.6228      0.2667   
ACEIX               0.0512  0.6616         -0.0022     -2.2986      0.9892   
JABAX               0.0650  0.5371         -0.0027     -3.1048      0.9990   

           R²  p(Alpha=0)  
Stock                      
CPODX  0.5970      0.2991  
WSHFX  0.9215      0.1375  
TMSIX  0.8660      0.2200  
ACEIX  0.8915      0.0042  
JABAX  0.8722      0.0000  


The extended market model is used to evaluate whether fund managers demonstrate market timing ability over the full sample period. This model extends the standard market model by including a squared market excess return term, capturing the curvature in the relationship between fund and market performance. The regression estimated for each fund is given by

$$
(R_{i,t} - R_{f,t}) = \alpha_i + \beta_i (R_{M,t} - R_{f,t}) + \gamma_i (R_{M,t} - R_{f,t})^2 + \varepsilon_t
$$

where $(R_{i,t} - R_{f,t})$ represents the fund’s excess return, $(R_{M,t} - R_{f,t})$ represents the market’s excess return, and $\alpha_i$, $\beta_i$, and $\gamma_i$ measure the fund’s selectivity, systematic risk exposure, and market timing ability, respectively. The parameter $\alpha_i$ (Jensen’s alpha) indicates whether the manager adds value through stock selection, $\beta_i$ measures sensitivity to market movements, and $\gamma_i$ captures whether the manager adjusts market exposure in response to changing conditions. A significantly positive $\gamma_i$ would suggest good market timing ability, while a value near zero would indicate little to no timing behavior.

The regression results from the full sample show that most funds have $\gamma_i$ coefficients that are small and statistically insignificant, suggesting limited evidence of strong market timing ability. However, a few funds exhibit slightly positive $\gamma_i$ estimates, which may point to minimal or weak timing behavior. This could indicate that some managers occasionally adjusted their market exposure in the right direction, but not consistently or significantly enough to demonstrate clear timing skill. Overall, the results imply that while most funds’ performance is largely explained by their systematic exposure to the market, there is a slight possibility that some managers engaged in limited market timing during the sample period.

***Statement of Collaboration (including ChatGPT)***: I collaborated with **Rosalia Mwidege** and **Theodore Ouyang**. Additionally, **ChatGPT** was used to debug any error-prone code and find the proper Excel-equivalent Python functions and libraries to properly execute the solutions to the problems, in conjunction with the hints listed on the problem set.

***Honor Code***: This assignment represents my own work in accordance with University regulations and class policy.